[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/01_web_crawling/01_web_crawling.ipynb)

# 01. 웹 크롤링 실습 (requests + BeautifulSoup + DB 저장)

> 관련 예제 프로젝트: [`example-projects/crawl-storage-example`](https://github.com/karzit/temp/tree/master/example-projects/crawl-storage-example) (A파트-1: 수집/저장) · 다른 라이브러리 선택지: [ALTERNATIVES.md](https://github.com/karzit/temp/blob/master/example-projects/crawl-storage-example/ALTERNATIVES.md)

## 이 장을 배우는 이유

이 시리즈의 목표는 **"우리 회사 규정을 물어보면 답해주는 챗봇"** 을 만드는 것입니다.

ChatGPT에게 "우리 회사 육아휴직은 몇 개월까지 쓸 수 있어?"라고 물으면 답을 못 합니다.
**그 회사 규정을 배운 적이 없기 때문**입니다. 그러면 방법은 하나뿐입니다.
**질문할 때 회사 규정 문서를 같이 넣어주는 것.**

그러려면 먼저 그 문서들이 **내 손에 있어야** 합니다. 사내 위키에, 게시판에, PDF 파일로
흩어져 있는 문서를 긁어모으는 것 — 그게 이번 장입니다. 파이프라인의 첫 단추입니다.

```
[01 크롤링] → 02 청킹 → 03 구조화 → 04 RAG 검색·답변 → 05 방어
  여기
```

이번 장에서 배우는 것

- `requests`로 웹 페이지 가져오기 (그리고 한글이 깨질 때 대처법)
- `BeautifulSoup`으로 HTML에서 **사람이 읽는 글자만** 뽑아내기
- `.env` 파일로 접속 정보를 코드와 분리하기
- 모은 문서를 DB에 저장하고, **같은 URL을 두 번 긁어도 중복이 안 생기게** 하기

**소요 시간**: 30~40분. 실제 웹사이트에 요청을 보내므로 인터넷 연결이 필요합니다.
(연습용으로 공개된 `toscrape.com`을 씁니다. [크롤링](https://github.com/karzit/temp/blob/master/glossary.md#crawling) 연습을 허용하는 사이트입니다.)

## 먼저 짚고 갈 것: 왜 원본을 그대로 DB에 넣나요?

크롤링한 걸 바로 검색엔진에 넣으면 안 될까요? 나중에 후회합니다.

크롤링은 **느리고 자주 실패합니다.** 문서 1000개를 긁는 데 몇십 분이 걸리고, 중간에
네트워크가 끊기기도 합니다. 그런데 나중에 "문서를 500자씩 자르지 말고 1000자씩 잘라볼까?"
하고 방식을 바꾸고 싶어집니다. **원본을 안 남겨뒀다면 그 몇십 분을 다시 써야 합니다.**

그래서 **원본(raw)은 원본대로 저장하고, 가공은 그 다음 단계에서** 합니다.
([원본/가공본 분리](https://github.com/karzit/temp/blob/master/glossary.md#raw-vs-processed)) 문제가 생겼을 때 "크롤링이 잘못됐나, 가공이
잘못됐나"를 가려내기도 쉬워집니다.

### 전체 흐름

```
URL 목록 -> requests로 페이지 요청
         -> BeautifulSoup으로 HTML 파싱 (본문 텍스트만 추출)
         -> (PDF 링크면) 바이트 그대로 저장
         -> DB에 UPSERT (같은 URL이면 덮어쓰기)
```

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요**(`Shift + Enter`).
- **실행 결과는 저장되어 있지 않습니다.** 직접 실행해야 출력이 나타납니다.
- 코드 셀 앞에는 **지금 무엇을 할 것인지**, 뒤에는 **결과를 어떻게 읽는지**를 적어두었습니다.
- `crawl.py`, `db.py`처럼 나오는 파일 이름은 예제 프로젝트
  [`crawl-storage-example`](https://github.com/karzit/temp/tree/master/example-projects/crawl-storage-example)의 실제 파일입니다.
  **열어보지 않아도 따라갈 수 있고**, 나란히 놓고 보면 더 좋습니다.
- 낯선 용어는 [glossary.md](https://github.com/karzit/temp/blob/master/glossary.md)에서 찾아보세요.
- **에러가 나거나 결과가 예상과 다르면** [troubleshooting.md](https://github.com/karzit/temp/blob/master/troubleshooting.md)를 먼저 보세요.
  설치 실패, 한글 깨짐, `NameError`, API 키, GPU 설정처럼 여러 노트북에서 반복되는 문제를 모아뒀습니다.

## 막혔을 때 — 이 노트북에서 자주 나오는 증상

| 증상 | 원인 | 해볼 것 |
|---|---|---|
| `ConnectionError` / `ReadTimeout` | 이 노트북은 **실제 웹사이트에 요청을 보냅니다** | 인터넷 연결 확인. 사내망이라면 프록시·방화벽이 `toscrape.com`을 막고 있을 수 있습니다 |
| `HTTPError: 403 Forbidden` | 사이트가 크롤러를 차단 | `HEADERS`의 `User-Agent`가 붙어 있는지 확인. 그래도 막히면 그 사이트는 크롤링을 허용하지 않는 것입니다 |
| `0권 파싱 완료` — 결과가 비어 있다 | 사이트 HTML 구조가 바뀌어 선택자가 안 맞음 | 브라우저에서 우클릭 → 검사로 실제 태그·클래스를 다시 확인 |
| 저장된 행의 **본문 길이가 0** | 자바스크립트로 내용을 그리는 사이트 (`requests`는 JS를 실행하지 않습니다) | 실습 5의 확인 셀에서 다루는 내용입니다. 그런 사이트는 Selenium/Playwright가 필요합니다 |
| `.env` 값을 고쳤는데 반영이 안 된다 | **`load_dotenv()`는 이미 설정된 환경변수를 덮어쓰지 않습니다** | `load_dotenv(override=True)`를 쓰거나 런타임을 재시작 |
| `sqlite3.OperationalError: no such column: binary_content` | 실습 6의 `ALTER TABLE` 셀을 건너뜀 | 그 셀을 먼저 실행 |
| `database is locked` | 같은 DB 파일에 열린 연결이 남아 있음 | 런타임 재시작. 실습 코드는 `with` 블록으로 매번 닫도록 되어 있습니다 |

여기 없는 문제(설치 실패, 한글 깨짐, API 키 설정 방법)는 [troubleshooting.md](https://github.com/karzit/temp/blob/master/troubleshooting.md)에 모아뒀습니다.

## 실습 0. 환경 설정

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q requests beautifulsoup4 python-dotenv

## 실습 1. `requests`로 페이지 가져오기

예제 프로젝트 `crawl-storage-example`의 `crawl.py`에 있는 `fetch()`와 같은 패턴입니다.
하는 일은 세 가지입니다 — 브라우저인 척하는 헤더를 붙이고, `timeout`을 걸고,
`raise_for_status()`로 실패를 바로 알아챕니다.

실전 팁이 하나 더 들어 있습니다. 서버가 응답 헤더에 `charset`을 알려주지 않으면
`requests`는 인코딩을 `ISO-8859-1`로 잘못 추측합니다. 그러면 `£`, `–` 같은 글자가
`Â£`처럼 깨집니다. `response.apparent_encoding`(본문 바이트를 보고 추정한 실제 인코딩)으로
보정하면 됩니다.

In [ ]:
import time
import requests

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; tutorial-crawler/1.0)"}


def fetch(url: str) -> requests.Response:
    # requests.get: 웹 페이지를 받아온다. timeout=10이면 10초 넘게 응답이 없을 때 포기
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()  # 200이 아니면 여기서 바로 예외를 던진다
    if response.encoding is None or response.encoding.lower() == "iso-8859-1":
        # charset 헤더가 없을 때 requests의 기본 추측(ISO-8859-1)이 틀리는 경우가 많아 보정한다
        response.encoding = response.apparent_encoding
    return response


resp = fetch("https://books.toscrape.com/")
print("status:", resp.status_code)
print("content-type:", resp.headers.get("Content-Type"))
print("본문 길이:", len(resp.text))
print(resp.text[:300])

**결과 읽는 법**

- **`status: 200`** 이면 정상입니다. 404면 없는 페이지, 403이면 접근 거부입니다.
  `raise_for_status()`가 없으면 404 페이지의 "Not Found" 화면을 **정상 문서인 줄 알고 저장하게 됩니다.**
  그래서 한 줄이지만 꼭 넣습니다.
- **본문 앞 300자가 `<!DOCTYPE html>...`** 로 시작합니다. 지금 받은 것은 사람이 보는 화면이 아니라
  **HTML 소스**입니다. 여기서 글자만 뽑아내는 것이 다음 실습입니다.
- 인코딩 보정 코드가 왜 필요한지도 짚고 갑니다. 서버가 "이 문서는 UTF-8이다"라고 알려주지 않으면
  `requests`는 유럽식 인코딩(`ISO-8859-1`)으로 넘겨짚습니다. 그러면 한글이 통째로 깨집니다.
  `apparent_encoding`은 **내용 바이트를 실제로 살펴보고** 인코딩을 추정한 값이라 훨씬 정확합니다.


## 실습 2. `BeautifulSoup`으로 목록 페이지 파싱하기

책 목록 페이지에서 제목과 가격만 뽑아냅니다. `.select()`는 CSS 선택자로 원하는 태그를 찾는 방법입니다.

In [ ]:
from bs4 import BeautifulSoup   # BeautifulSoup: HTML 문자열을 태그 구조로 파싱해 원하는 부분만 뽑게 해준다

soup = BeautifulSoup(resp.text, "html.parser")

books = []
for article in soup.select("article.product_pod"):
    title = article.h3.a["title"]
    price = article.select_one("p.price_color").get_text(strip=True)
    books.append({"title": title, "price": price})

print(f"{len(books)}권 파싱 완료")
for b in books[:5]:
    print(b)

**결과 읽는 법** — 책 20권의 제목과 가격이 뽑혔습니다. HTML 소스가 **파이썬 딕셔너리 목록**으로
바뀐 것입니다. 이제부터는 웹페이지가 아니라 데이터로 다룰 수 있습니다.

**`select`와 `select_one`을 읽는 법**

| 코드 | 뜻 |
|---|---|
| `soup.select("article.product_pod")` | `<article class="product_pod">` 태그를 **전부** 찾기 |
| `article.h3.a["title"]` | 그 안의 `<h3>` 안의 `<a>` 태그의 `title` 속성값 |
| `article.select_one("p.price_color")` | 클래스가 `price_color`인 `<p>` **하나** |
| `.get_text(strip=True)` | 태그를 벗기고 글자만, 앞뒤 공백 제거 |

**이 선택자는 어떻게 알아냈을까요?** 브라우저에서 해당 부분을 우클릭 → "검사"를 누르면
그 요소의 태그와 클래스가 보입니다. 크롤링 코드를 쓸 때는 항상 이 과정을 먼저 거칩니다.


## 실습 3. `crawl.py`의 `extract_text_from_html` 재구현

`crawl.py`의 `extract_text_from_html()`은 HTML에서 `<script>`/`<style>` 태그를 걷어내고,
본문 글자만 줄바꿈 기준으로 정리해 뽑아냅니다.
그 함수를 그대로 만들어보고 quotes.toscrape.com 페이지에 적용해봅니다.

In [ ]:
def extract_text_from_html(html: str) -> str:
    """HTML에서 사람이 읽는 본문 글자만 뽑아낸다 (crawl.py와 동일한 로직)."""
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    lines = [line.strip() for line in text.splitlines()]
    return "\n".join(line for line in lines if line)


quotes_resp = fetch("https://quotes.toscrape.com/")
clean_text = extract_text_from_html(quotes_resp.text)
print(clean_text[:500])

**결과 읽는 법** — 이번에는 태그가 하나도 없이 **읽을 수 있는 문장만** 나옵니다.

세 가지 처리가 들어갔습니다.

1. **`<script>`, `<style>` 제거** — 자바스크립트 코드와 CSS는 사람이 읽는 내용이 아닙니다.
   이걸 안 지우면 검색 결과에 코드 조각이 섞여 나옵니다.
2. **`get_text(separator="\n")`** — 태그를 벗기되 태그 경계에서 줄을 바꿔, 문장이 서로 붙지 않게 합니다.
3. **빈 줄 제거** — HTML에는 의미 없는 공백이 아주 많습니다. 그대로 두면 나중에 잘랐을 때
   **내용은 없고 공백만 든 조각**이 생깁니다.

이 셋을 안 하면 다음 단계([청킹](https://github.com/karzit/temp/blob/master/glossary.md#chunking), 검색)의 품질이 통째로 나빠집니다.
**"쓰레기를 넣으면 쓰레기가 나온다"** 는 말이 [RAG](https://github.com/karzit/temp/blob/master/glossary.md#rag)에서 특히 잘 맞습니다.


## 실습 4. `python-dotenv`로 설정값 관리하기

API 키나 접속 정보처럼 코드에 직접 적으면 안 되는 값은 `.env` 파일에 따로 두고 `load_dotenv()`로 불러옵니다. Colab에서는 파일을 직접 만들어 확인해봅니다.

In [ ]:
with open(".env", "w", encoding="utf-8") as f:
    f.write("DATABASE_URL=sqlite:///crawled.db\n")
    f.write("CRAWL_DELAY_SECONDS=0.5\n")

from dotenv import load_dotenv
import os

load_dotenv()

# config.py와 동일한 패턴: os.getenv(키, 기본값)
DATABASE_URL = os.getenv("DATABASE_URL", "sqlite:///default.db")
CRAWL_DELAY_SECONDS = float(os.getenv("CRAWL_DELAY_SECONDS", "1.0"))

print("DATABASE_URL:", DATABASE_URL)
print("CRAWL_DELAY_SECONDS:", CRAWL_DELAY_SECONDS)

**결과 읽는 법** — `.env` 파일에 적어둔 값이 그대로 읽혔습니다.

**왜 이렇게 할까요?** DB 비밀번호나 API 키를 코드에 직접 적으면 그대로 GitHub에 올라갑니다.
공개 저장소에 올린 API 키는 **몇 분 만에 발견되어 도용됩니다.** 실제로 흔한 사고입니다.

그래서 이렇게 나눕니다.

- **코드**: `os.getenv("DATABASE_URL", "기본값")` — 값이 아니라 **이름만** 적혀 있음 → 커밋해도 안전
- **`.env` 파일**: 실제 값이 들어 있음 → `.gitignore`에 넣어 **절대 커밋하지 않음**

`os.getenv(키, 기본값)`의 두 번째 인자는 **`.env`가 없을 때 쓸 값**입니다.
덕분에 설정 파일이 없어도 프로그램이 죽지 않습니다. ([python-dotenv](https://github.com/karzit/temp/blob/master/glossary.md#dotenv))


## 실습 5. `sqlite3`로 크롤링 결과 저장하기

`db.py`의 `CREATE_TABLE_SQL`/`UPSERT_SQL` 패턴을 sqlite3 문법으로 옮깁니다.

`ON CONFLICT ... DO UPDATE`(UPSERT)는 ANSI SQL 표준은 아니지만,
[PostgreSQL](https://github.com/karzit/temp/blob/master/glossary.md#postgresql)과 SQLite가 공통으로 지원하는 확장 문법입니다.
덕분에 여기서 쓴 sqlite3 코드가 실제 PostgreSQL에서도 거의 그대로 동작합니다.

In [ ]:
import sqlite3

CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS crawled_documents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    url TEXT NOT NULL UNIQUE,
    content_type TEXT NOT NULL,
    text_content TEXT,
    crawled_at TEXT NOT NULL DEFAULT (datetime('now'))
);
"""

UPSERT_SQL = """
INSERT INTO crawled_documents (url, content_type, text_content, crawled_at)
VALUES (?, ?, ?, datetime('now'))
ON CONFLICT(url) DO UPDATE SET
    content_type = excluded.content_type,
    text_content = excluded.text_content,
    crawled_at = datetime('now');
"""


def init_db(db_path: str):
    with sqlite3.connect(db_path) as conn:   # sqlite3: 파일 하나로 동작하는 내장 DB. 서버를 띄울 필요가 없다
        conn.execute(CREATE_TABLE_SQL)


def save_document(db_path: str, url: str, content_type: str, text_content: str):
    with sqlite3.connect(db_path) as conn:
        conn.execute(UPSERT_SQL, (url, content_type, text_content))


DB_PATH = "crawled.db"
init_db(DB_PATH)
print("테이블 준비 완료")

**결과 읽는 법** — 아직 눈에 보이는 건 없지만, `crawled.db` 파일이 만들어지고 그 안에
테이블이 생겼습니다. [sqlite3](https://github.com/karzit/temp/blob/master/glossary.md#sqlite3)는 **서버를 띄울 필요 없이 파일 하나로 동작하는 DB**라
실습에 딱 좋습니다.

**SQL에서 꼭 볼 두 곳이 있습니다.**

1. **`url TEXT NOT NULL UNIQUE`** — URL이 중복될 수 없게 못을 박았습니다.
2. **`ON CONFLICT(url) DO UPDATE SET ...`** — 그 못에 걸렸을 때(=같은 URL이 이미 있을 때)
   **에러를 내지 말고 내용을 갱신하라**는 뜻입니다. 이것이 [UPSERT](https://github.com/karzit/temp/blob/master/glossary.md#upsert)입니다.

이게 없으면 크롤러를 두 번 돌릴 때마다 똑같은 문서가 계속 쌓입니다.
그러면 검색 결과에 같은 내용이 세 번씩 나오게 됩니다. **크롤러는 재실행이 잦기 때문에
UPSERT는 선택이 아니라 필수입니다.**


이제 `crawl.py`의 `main()`과 같은 방식으로 여러 URL을 순회하며 저장합니다. 요청 사이에 `CRAWL_DELAY_SECONDS`만큼 쉬어가는 것도 그대로 재현합니다.

In [ ]:
CRAWL_TARGETS = [
    "https://quotes.toscrape.com/",
    "https://quotes.toscrape.com/page/2/",
    "https://quotes.toscrape.com/tag/love/",
]

for url in CRAWL_TARGETS:
    try:
        print(f"크롤링 중: {url}")
        page = fetch(url)
        text = extract_text_from_html(page.text)
        save_document(DB_PATH, url, "html", text)
    except requests.RequestException as e:
        print(f"실패: {url} ({e})")

    time.sleep(CRAWL_DELAY_SECONDS)

print("크롤링 완료")

**결과 읽는 법** — 세 페이지를 차례로 긁어 저장했습니다. 두 가지를 눈여겨보세요.

**① `try` / `except`로 감싼 이유**
URL 하나가 실패했다고 전체가 멈추면 안 됩니다. 1000개 중 3개가 죽었다고 997개를 버릴 수는 없으니까요.
실패한 것만 기록하고 넘어갑니다. **크롤링은 실패를 전제로 짜는 코드입니다.**

**② `time.sleep`으로 쉬어가는 이유**
쉬지 않고 요청을 퍼부으면 상대 서버에 부담을 주고, 공격으로 오인받아 IP가 차단됩니다.
많은 사이트가 [robots.txt](https://github.com/karzit/temp/blob/master/glossary.md#robots-txt)에 `Crawl-delay`로 "이만큼 쉬어달라"고 적어둡니다.
**남의 서버를 쓰는 일이니 예의를 지키는 것**이고, 동시에 내 크롤러를 오래 살리는 방법이기도 합니다.

크롤링 전에 대상 사이트의 `robots.txt`(예: `https://example.com/robots.txt`)와 이용약관을
확인하는 습관을 들이세요.


### 저장 결과 확인

크롤링이 끝났으니 **DB에 실제로 무엇이 들어갔는지** 확인합니다.
`SELECT`로 id·URL·문서 타입·본문 길이·수집 시각을 뽑아봅니다. 본문 길이가 0인 행이 있다면
그 페이지는 파싱에 실패한 것입니다.

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    rows = conn.execute(
        "SELECT id, url, content_type, length(text_content), crawled_at FROM crawled_documents"
    ).fetchall()   # fetchall(): SELECT 결과를 전부 리스트로 가져온다

for row in rows:
    print(row)

**결과 읽는 법** — 저장된 행이 3개 나옵니다. 각 행은 `(id, url, 문서타입, 본문 길이, 수집 시각)`입니다.

**여기서 반드시 확인할 것: 본문 길이가 0인 행이 있는지.**
길이가 0이면 페이지는 받았는데 **글자를 못 뽑은 것**입니다. 자바스크립트로 내용을 그리는
사이트에서 흔히 생깁니다(`requests`는 자바스크립트를 실행하지 않습니다).
그런 사이트는 Selenium이나 Playwright 같은 도구가 필요합니다.

**저장한 다음에는 항상 이렇게 꺼내서 확인하세요.** "저장했다"는 메시지가 떴다고
제대로 들어간 것은 아닙니다.


### 참고: 실제 `psycopg2` 버전

DB 서버(PostgreSQL)가 있다면 위 코드는 아래처럼 거의 그대로 옮겨갑니다 (`db.py` 원본):

```python
import psycopg2

def get_connection():
    return psycopg2.connect(DATABASE_URL)

def save_document(url, content_type, text_content, binary_content):
    with get_connection() as conn:
        with conn.cursor() as cur:
            binary_param = psycopg2.Binary(binary_content) if binary_content is not None else None
            cur.execute(UPSERT_SQL, (url, content_type, text_content, binary_param))
        conn.commit()
```
핵심 차이는 두 가지뿐입니다. 파라미터 자리표시자가 sqlite3는 `?`, psycopg2는 `%s`라는 것,
그리고 psycopg2는 바이너리(PDF 등)를 넣을 때 `psycopg2.Binary()`로 감싸야 한다는 것입니다.

## 실습 6. PDF처럼 바이너리 데이터는 다르게 저장하기

`crawl_one()`은 URL이 `.pdf`로 끝나면 텍스트로 바꾸지 않고 **원본 바이트 그대로** 저장합니다.
PDF 안의 글자를 뽑는 일은 다음 단계인 청킹 노트북의 몫입니다.
sqlite3의 `BLOB` 타입에 바이너리를 저장하는 것도 함께 연습해봅니다.

In [ ]:
ALTER_SQL = "ALTER TABLE crawled_documents ADD COLUMN binary_content BLOB"
with sqlite3.connect(DB_PATH) as conn:
    try:
        conn.execute(ALTER_SQL)
    except sqlite3.OperationalError:
        pass  # 컬럼이 이미 있으면 무시


def crawl_one(url: str) -> None:
    response = fetch(url)
    if url.lower().endswith(".pdf"):
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute(
                "INSERT INTO crawled_documents (url, content_type, binary_content, crawled_at) "
                "VALUES (?, 'pdf', ?, datetime('now')) "
                "ON CONFLICT(url) DO UPDATE SET binary_content = excluded.binary_content",
                (url, sqlite3.Binary(response.content)),
            )
    else:
        text = extract_text_from_html(response.text)
        save_document(DB_PATH, url, "html", text)


# PDF 링크는 실제로 호출하지 않고, 분기 로직만 확인합니다.
print("'a.pdf' -> pdf 분기:", "a.pdf".lower().endswith(".pdf"))
print("'a.html' -> pdf 분기:", "a.html".lower().endswith(".pdf"))

**결과 읽는 법** — PDF 링크는 **글자를 뽑지 않고 바이트 그대로** 저장합니다. 왜일까요?

PDF에서 글자를 뽑는 건 생각보다 까다롭습니다. 표가 깨지고, 2단 편집이 섞이고,
스캔한 PDF는 아예 이미지라 글자가 없습니다. **이 처리를 여기서 하면 방식을 바꿀 때마다
PDF를 다시 내려받아야 합니다.**

그래서 원본만 저장해두고, 글자 추출은 다음 노트북(청킹)에서 합니다.
도입부에서 말한 **"원본은 원본대로"** 원칙이 여기에도 그대로 적용됩니다.

`sqlite3.Binary(...)`로 감싸는 것은 "이건 글자가 아니라 바이너리 덩어리"라고 알려주는 것입니다.
PostgreSQL에서는 `psycopg2.Binary(...)`가 같은 역할을 합니다.


## 정리

이번 장에서 한 일

1. `requests`로 페이지를 받아오고, **인코딩이 깨질 때 보정**하는 법을 배웠습니다
2. `BeautifulSoup`으로 `<script>`/`<style>`을 걷어내고 **본문 글자만** 뽑았습니다
3. `.env`로 접속 정보를 코드에서 분리했습니다
4. sqlite3에 저장하되, **UPSERT로 같은 URL의 중복을 막았습니다**
5. PDF 같은 바이너리는 텍스트로 바꾸지 않고 **원본 그대로** 두었습니다

**가장 기억할 것**: **원본은 원본대로 남긴다.** 가공은 언제든 다시 할 수 있지만,
크롤링은 다시 하기 어렵습니다.

**스스로 확인해보기**

- [ ] `raise_for_status()`가 없으면 무엇이 문제인지 안다
- [ ] `<script>` 태그를 지우고 본문을 뽑는 이유를 설명할 수 있다
- [ ] `.env`에 접속 정보를 두는 이유를 안다
- [ ] UPSERT가 없으면 같은 URL을 두 번 긁었을 때 무슨 일이 생기는지 안다
- [ ] 요청 사이에 `time.sleep`을 넣는 이유를 안다

## 연습 문제

1. `CRAWL_TARGETS`를 `quotes.toscrape.com`의 태그 페이지 3~4개로 확장해 순회하고,
   각 페이지의 명언(quote) 개수를 세어 출력해보세요.
2. 같은 URL을 두 번 크롤링해도 테이블에 중복 행이 생기지 않는지 직접 확인해보세요.
   `crawled_at` 값이 두 번째 실행에서 갱신되는지도 함께 보세요.
3. 일부러 없는 주소(예: `https://quotes.toscrape.com/없는페이지`)를 `CRAWL_TARGETS`에 넣고
   실행해보세요. 프로그램이 멈추지 않고 넘어가나요? 그게 왜 중요할까요?

**해설/정답**: [01_web_crawling_solutions.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/01_web_crawling/01_web_crawling_solutions.ipynb)

## 다음 단계

다음 노트북([02_text_chunking](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/02_text_chunking/02_text_chunking.ipynb))에서는 이렇게 모은
원본을 **검색에 쓸 수 있도록 잘게 자릅니다.** 문서 한 편을 통째로 AI에게 던질 수 없는 이유부터
시작합니다.
